# Factor-neutral sizing: does the convexity edge survive the hedge?

**The question.** The cross-strategy attribution found that the long-end
flatteners are not sized to what they claim to trade, and the inference was
*"we are sizing these incorrectly"*. This notebook tests that constructively.
It builds the alternative sizings, runs them over the same window, the same 91
cohorts and the same engine, and asks the only question that decides it:

> **after neutralising the dominant factor FOR THAT STRUCTURE, does a positive
> convexity edge remain, or does the P&L simply go to zero?**

**The answer, up front: the edge survives on the three tight forward pairs and
it dies on 5Y/30Y** — and the split is exactly the one the attribution
predicted, structure by structure.

| structure | dominant factor | incumbent | after neutralising it | retained |
|---|---|---|---|---|
| 30Y/50Y | level | 341.0 bp | **300.7 bp** | 88% |
| 20Yx5Y/25Yx5Y | level | 479.6 bp | **426.3 bp** | 89% |
| 10Yx10Y/20Yx10Y | level | 500.1 bp | **334.8 bp** | 67% |
| 5Y/30Y | slope | 268.8 bp | **−739.8 bp** | −275% |

All three tight pairs stay positive, keep a 53–73% hit rate, and keep the
convexity term intact at `t = 4.1–4.9` with an incremental R² of 0.32–0.36 —
statistically indistinguishable from the incumbent's. Neutralising slope on
5Y/30Y takes it from **+269 bp to −740 bp** and the hit rate from 0.53 to 0.20.
That book's P&L *was* the slope bet, and there is nothing underneath it.

Three findings sit alongside that and are not decoration:

1. **The hedges do what they claim.** Measured on realised daily P&L, the level
   regressor's incremental R² falls 0.162 → 0.019 on 20Yx5Y/25Yx5Y, 0.215 →
   0.073 on 10Yx10Y/20Yx10Y and 0.0111 → 0.0005 on 30Y/50Y; the slope
   regressor's falls **0.883 → 0.012** on 5Y/30Y. That is a measurement on the
   books, not a property of the solve, and it is the half of the answer that
   does not depend on a small sample.
2. **`slope_beta_hedged` beats the incumbent on three of four structures** —
   1.38x, 1.82x and 1.86x — with its overlay charged at the most expensive
   honest reading. But read *why*: the flatteners were **losing** on slope
   (share_slope = −149% on 5Y/30Y), so removing it adds money. That is a
   genuine risk reduction whose *size* is a property of what slope did in
   2019–2026, not a new source of return.
3. **`pc1_neutral` on 5Y/30Y is an estimation artefact, not an edge.** It scores
   +1408 bp walk-forward against **−66 bp** on the same rule with full-sample
   loadings — a gap of 105% of its own total. That is the PCA moving, not a
   trade, and it is reported here rather than banked.

**On the multiple-testing bar:** four distinct sizings at a measured pooled
effective sample of 9.79 observations sets `E[max Sharpe | null] = 0.3327` per
observation. Seven books clear it — but four of them are the *same structure*
(20Yx5Y/25Yx5Y), whose incumbent already cleared at 0.9333, so re-sizing is not
what produced them. Section 13 states exactly what does and does not survive.

## 0. CONFIG

Everything the run depends on, in one place. The BACKTEST knobs are
deliberately not restated here — they come from
`strat1_longend_listed.strat1_config()`, strategy 1's committed base run
($100k package DV01, monthly cohorts, 1-year hold, 0.5 bp one way, no
force-close, 2019-01-02..2026-08-14). The whole point of this study is that its
books are directly comparable to that run, and a single differing knob would
make every comparison below a comparison of two configurations instead of two
sizings.

In [1]:
from __future__ import annotations

import os

os.environ.setdefault("ARBS_SUPABASE_ENABLED", "0")

import json
import pathlib
import sys
import time

import numpy as np
import pandas as pd
import plotly.io as pio

pio.renderers.default = "plotly_mimetype+notebook_connected"

_REPO = pathlib.Path.cwd()
while not (_REPO / "RVUtils").exists() and _REPO != _REPO.parent:
    _REPO = _REPO.parent
sys.path.insert(0, str(_REPO))

from BT.trade_dashboard import compare_curves, summary_stats, trade_dashboard
from RVUtils.ConvexityRV import factor_attribution as fa
from RVUtils.ConvexityRV import factor_neutral_sizing as fns
from RVUtils.ConvexityRV import strat1_longend_listed as ll
from RVUtils.ConvexityRV import strat1_threeway as tw

pd.set_option("display.width", 260)
pd.set_option("display.max_columns", 90)

DATA = _REPO / "notebooks" / "data" / "convexity_rv"
CFG = fns.FactorNeutralConfig()
S1 = ll.strat1_config()
FCFG = fa.FactorConfig()
DV01 = float(S1.package_dv01)
T_START = time.time()

print(f"repo        {_REPO}")
print(f"curve       {S1.curve}   {S1.start} .. {S1.end}")
print(f"package     ${DV01:,.0f} DV01, {S1.cohort_freq} cohorts, "
      f"{S1.horizon} hold, {S1.cost_bp_one_way} bp one way")
print(f"sizings     {CFG.sizings}")
print(f"scored      {CFG.scored_sizings}  ({len(CFG.scored_sizings)} trials)")
print(f"weights     {CFG.weight_mode}, min_fit_days {CFG.min_fit_days}, "
      f"hard floor {CFG.hard_fit_floor}, beta window {CFG.beta_window}")
print(f"hedge leg   {CFG.hedge_leg}   slope overlay {fns.SLOPE_INSTRUMENT}")
print(f"legs        {CFG.legs}")
print(f"certify     {CFG.certify_structure} x {CFG.certify_sizing}")

C:\Users\chris\clee\ARBS-cvx\RVUtils\ConvexityRV\curve_ops.py:61: LicenceNotice:


Rateslib is source-available (not open-source) software distributed under a dual-licence model.
No commercial licence is registered for this installation. Use is therefore permitted for non-commercial purposes only (at-home or university based academic use).
Any use in commercial, professional, or for-profit environments, including evaluation or trial use, requires a valid commercial licence or an approved evaluation licence.
Certain features may require a registered commercial or evaluation licence in current or future versions.
For licensing information or to register a licence, please visit: https://rateslib.com/licence



repo        C:\Users\chris\clee\ARBS-cvx
curve       USD-SOFR-1D   2019-01-01 .. 2026-08-14
package     $100,000 DV01, monthly cohorts, 1Y hold, 0.5 bp one way
sizings     ('dv01_neutral', 'pc1_neutral', 'pc12_neutral', 'slope_beta_hedged')
scored      ('dv01_neutral', 'pc1_neutral', 'pc12_neutral', 'slope_beta_hedged')  (4 trials)
weights     walk_forward, min_fit_days 250, hard floor 15, beta window 252
hedge leg   10Y   slope overlay (('5Y', 1.0), ('30Y', -1.0))
legs        ('50Y', '30Y', '10Y', '5Y', '25Yx5Y', '20Yx5Y', '20Yx10Y', '10Yx10Y')
certify     5Y/30Y x pc12_neutral


## 1. The retarget — why the default sizing in the draft was wrong

This study was drafted to test one claim: *DV01-neutral sizing leaves a
dominant **slope** bet*. The attribution then finished, and reading the same
four packages as a share of their **own factor variance** rather than as a
dollar exposure inverted the premise for three of the four:

| structure | level | slope | curvature | dominant |
|---|---|---|---|---|
| 5Y/30Y | 5.1% | **88.9%** | 6.0% | slope |
| 30Y/50Y | **43.3%** | 29.3% | 27.4% | level |
| 20Yx5Y/25Yx5Y | **69.0%** | 14.7% | 16.4% | level |
| 10Yx10Y/20Yx10Y | **61.3%** | 0.2% | 38.5% | level |

The slope premise is true for **5Y/30Y and nothing else**. For the three tight
forward pairs the dominant unwanted exposure is **level** — in a construction
whose entire stated purpose was to remove level exposure.

**Why DV01-neutrality does not remove level.** It would, if PC1's loadings were
flat. They are not: PC1 is humped, 0.279 at 2Y, 0.334 at 7Y, 0.272 at 50Y. Two
legs of equal and opposite DV01 therefore leave `dv01 * (v1_front − v1_back)`,
and a *tight* pair has small slope and curvature differences, so that small
absolute residue is a *large share* of the little exposure the package has.

So `pc1_neutral` is added as a first-class sizing and is the headline test for
the three tight pairs. It is a two-leg trade — one constraint plus a
normalisation — and it is deliberately **not** DV01-neutral. That is the point.

The table above is copied from the attribution report, and copied numbers rot,
so the next cell recomputes all of it from the shared `FactorModel` and prints
the gap.

In [2]:
PANEL = pd.read_parquet(DATA / "factor_rate_panel.parquet")
PANEL.index = pd.to_datetime(PANEL.index)
FM = fa.fit_factor_model(PANEL, FCFG)
CALENDAR = FM.scores.index

print(f"rate panel  {PANEL.shape[0]} days x {PANEL.shape[1]} tenors, "
      f"{PANEL.index.min():%Y-%m-%d} .. {PANEL.index.max():%Y-%m-%d}")
print(f"factor fit  {FM.rates_bp.shape[0]} levels -> {FM.scores.shape[0]} changes")
print(f"explained   {FM.explained_variance.head(3).round(4).to_dict()}")
print()
print("PC labels are TESTED structurally, never assumed from eigenvalue order:")
print(fa.classify_pcs(FM.loadings, FM.tenors).to_string())

rate panel  1904 days x 12 tenors, 2019-01-02 .. 2026-08-12
factor fit  1904 levels -> 1903 changes
explained   {'PC1': 0.8833, 'PC2': 0.1071, 'PC3': 0.0073}

PC labels are TESTED structurally, never assumed from eigenvalue order:
         label  same_sign_frac  n_sign_flips  corr_with_log_tenor  loading_mean  loading_sum  short_end  long_end
pc                                                                                                               
PC1      level        1.000000             0            -0.459506      0.300741     3.308156   0.279402  0.272067
PC2      slope        0.636364             1            -0.980402     -0.007751    -0.085259   0.542921 -0.305611
PC3  curvature        0.636364             2             0.096711      0.019950     0.219445   0.565951  0.280249


In [3]:
DOMINANT = fns.dominant_factor_table(FM)
print("Re-measured on the shared basis, against the attribution report's numbers:")
print(DOMINANT[["structure", "var_level", "var_slope", "var_curv",
                "prior_var_level", "prior_var_slope", "prior_var_curv",
                "max_abs_prior_gap", "dominant", "headline_sizing",
                "pc1_leakage_vs_outright"]].round(4).to_string(index=False))
_gap = float(DOMINANT["max_abs_prior_gap"].max())
print(f"\nlargest disagreement with the attribution report: {_gap:.2e} "
      f"({'CONFIRMED' if _gap < 0.005 else 'DISAGREES -- stop and re-read'})")
print("\n`pc1_leakage_vs_outright` is the quieter half of the finding: the share")
print("of a SAME-SIZE outright's level exposure that survives DV01-neutrality.")
print("3.8%-16.7%. Small in dollars, and for a tight pair it is most of the risk.")

Re-measured on the shared basis, against the attribution report's numbers:
      structure  var_level  var_slope  var_curv  prior_var_level  prior_var_slope  prior_var_curv  max_abs_prior_gap dominant headline_sizing  pc1_leakage_vs_outright
         5Y/30Y     0.0514     0.8886    0.0600            0.051            0.889           0.060             0.0004    slope    pc12_neutral                   0.1668
        30Y/50Y     0.4327     0.2932    0.2741            0.433            0.293           0.274             0.0003    level     pc1_neutral                   0.0383
  20Yx5Y/25Yx5Y     0.6898     0.1466    0.1635            0.690            0.147           0.164             0.0005    level     pc1_neutral                   0.0725
10Yx10Y/20Yx10Y     0.6134     0.0021    0.3845            0.613            0.002           0.385             0.0005    level     pc1_neutral                   0.1033

largest disagreement with the attribution report: 5.14e-04 (CONFIRMED)

`pc1_leakage_vs_o

## 2. One engine pass per LEG, and why that is exact rather than approximate

A package is a weighted sum of legs, and swap NPV is linear in `bpv`. So if the
engine is run once per LEG at a fixed unit DV01, with each cohort carrying its
own tag, then **every sizing of every structure is a linear combination of the
same eight mark matrices**:

```
equity_s(t) = Σ_k Σ_L (r_{s,k,L} / UNIT_DV01) · contrib_{L,k}(t) − fees_s(t)

contrib_{L,k}(t) = mark_{L,k}(t)          while the leg is open
                 = gross_realized_{L,k}   once it has unwound
```

That is not an approximation of the package run, it *is* the package run:
`resolve_pricable` solves notional off pv01 linearly, `rl.IRS.npv` is linear in
notional, and no cash is realised mid-hold. Section 3 measures that claim
against four genuine engine runs rather than asserting it.

The payoff is that a sizing comparison costs no extra engine time. Sixteen
books, four sizings, walk-forward weights that change every month, and a
monthly-rebalanced overlay: all of them are re-weightings of eight fixed mark
matrices. Nothing in the comparison is a re-run, so nothing in it can differ
for a reason other than the weights — which is the whole point of a sizing
experiment.

### The one grid wrinkle, measured rather than assumed

The engine's `mtm_history` carries a day only if the curve priced that leg on
it. On **2019-04-19 (Good Friday)** the local SOFR curve prices 5Y and 25Yx5Y
but not 10Y, 30Y or 50Y. Requiring bit-identical indices would refuse a
perfectly usable cache; unioning them would forward-fill a mark that was never
taken. So the legs are **intersected**, which lands on 1907 days — precisely
the grid the four stored two-leg engine runs use, so section 3 compares like
with like. No cohort entry or exit falls on the dropped day; that is asserted,
not hoped.

In [4]:
LEGS, IDX = fns.load_leg_runs(DATA, log=print)
SCHED = LEGS["5Y"].cohorts
ENTRIES = list(pd.to_datetime(SCHED["entry"]))
NLIVE = fns.live_packages(SCHED, IDX)

print(f"\n{len(LEGS)} legs on a common grid of {len(IDX)} marks, "
      f"{IDX.min():%Y-%m-%d} .. {IDX.max():%Y-%m-%d}")
print(f"{len(SCHED)} cohorts, {int(SCHED['closed'].sum())} closed, "
      f"{int((~SCHED['closed']).sum())} still live at the end")
print(f"live packages: mean {NLIVE.mean():.2f}, max {NLIVE.max():.0f}")
print()
print(pd.DataFrame({
    lg: {"terminal_usd": float(r.equity.iloc[-1]),
         "n_marks": int(len(r.equity)),
         "n_closed": int(r.cohorts["closed"].sum()),
         "mean_gross_bp": float(r.cohorts["gross_pnl_bp"].mean())}
    for lg, r in LEGS.items()}).T.to_string())

leg grids intersected to 1907 marks; dropped ['2019-04-19'] (priced for some legs only)

8 legs on a common grid of 1907 marks, 2019-01-02 .. 2026-08-14
91 cohorts, 79 closed, 12 still live at the end
live packages: mean 11.16, max 13

         terminal_usd  n_marks  n_closed  mean_gross_bp
50Y      2.042665e+08   1907.0      79.0      20.427934
30Y      2.463131e+08   1907.0      79.0      25.744724
10Y      2.489179e+08   1907.0      79.0      25.744593
5Y       2.917110e+08   1907.0      79.0      30.147292
25Yx5Y   1.898091e+08   1907.0      79.0      18.580415
20Yx5Y   2.471182e+08   1907.0      79.0      25.651776
20Yx10Y  2.209368e+08   1907.0      79.0      22.393349
10Yx10Y  2.745574e+08   1907.0      79.0      29.723676


In [5]:
GREEKS = fns.load_leg_greeks(DATA)
GAMMA = fns.leg_gamma_bp(GREEKS)
CARRY = fns.leg_carry_bp(GREEKS)
print(f"greeks: {len(GREEKS)} rows, {GREEKS['leg'].nunique()} legs x "
      f"{GREEKS['date'].nunique()} cohort entry dates")
print(pd.DataFrame({"gamma_usd_per_bp2_per_dv01": GAMMA,
                    "carry_bp_1y": CARRY,
                    "rate_pct": GREEKS.groupby("leg")["rate_pct"].mean()}).round(6).to_string())

greeks: 1274 rows, 14 legs x 91 cohort entry dates
         gamma_usd_per_bp2_per_dv01  carry_bp_1y  rate_pct
leg                                                       
10Y                       -0.000996     0.584646  0.026169
10Yx10Y                   -0.003092    -1.712595  0.029401
15Y                       -0.001449     0.888631  0.027249
20Y                       -0.001880    -0.325859  0.027573
20Yx10Y                   -0.005176    -1.182738  0.024371
20Yx5Y                    -0.004677    -0.009975  0.025436
25Y                       -0.002307    -1.021358  0.027288
25Yx5Y                    -0.005708    -0.011382  0.023138
2Y                        -0.000211   -30.930945  0.026022
30Y                       -0.002727    -1.513789  0.026851
3Y                        -0.000313   -21.071748  0.025171
50Y                       -0.004501    -2.874022  0.024280
5Y                        -0.000514    -7.100792  0.024913
7Y                        -0.000711    -1.840889  0.025359


**Carry tie-out.** The greeks pass is a fresh repricing loop and nothing above
depends on it being right unless it is checked. The DV01-neutral package's
carry, rebuilt from the per-leg numbers as `carry_front − carry_back`, must
reproduce `carry_roll_bp` on the committed signal panel exactly — same
quantity, independent code path.

In [6]:
_sp = pd.read_parquet(DATA / "strat1_signal_panel.parquet")
_sp["date"] = pd.to_datetime(_sp["date"])
_rows = []
for _lab, _f, _b in fa.STRAT1_STRUCTURES:
    _ref = _sp[_sp["structure"] == _lab].set_index("date")["carry_roll_bp"]
    _mine = pd.Series({_d: float(fns.leg_carry_bp(GREEKS, on=_d).get(_f, np.nan)
                                 - fns.leg_carry_bp(GREEKS, on=_d).get(_b, np.nan))
                       for _d in ENTRIES})
    _r = _ref.reindex(_mine.index)
    _rows.append({"structure": _lab, "greeks_mean_bp": _mine.mean(),
                  "panel_mean_bp": _r.mean(), "corr": _mine.corr(_r),
                  "max_abs_gap_bp": float((_mine - _r).abs().max())})
CARRY_TIEOUT = pd.DataFrame(_rows)
print(CARRY_TIEOUT.round(6).to_string(index=False))
assert float(CARRY_TIEOUT["max_abs_gap_bp"].max()) < 1e-9, "greeks carry does not tie out"
print("\nExact on every structure. The greeks pass is measuring the same object")
print("the committed signal panel measures, so the hedge's carry is real.")

      structure  greeks_mean_bp  panel_mean_bp  corr  max_abs_gap_bp
         5Y/30Y       -5.587003      -5.587003   1.0             0.0
        30Y/50Y        1.360233       1.360233   1.0             0.0
  20Yx5Y/25Yx5Y        0.001407       0.001407   1.0             0.0
10Yx10Y/20Yx10Y       -0.529857      -0.529857   1.0             0.0

Exact on every structure. The greeks pass is measuring the same object
the committed signal panel measures, so the hedge's carry is real.


## 3. Certification I — the composition against four genuine engine runs

`strat1_le_unit_equity_*.parquet` are four real `QueryDrivenBacktest` passes of
the two-leg DV01-neutral package, produced by a different module before this
one existed. Rebuilding them as `front − back` out of the per-leg mark
matrices, and charging the fee with the **generalised per-leg cost model**
rather than the engine's flat per-cohort figure, tests three things at once:

* that a package IS the sum of its legs (the linearity the whole design rests on);
* that the leg runs share the stored runs' cohort schedule to the day;
* that `2 · (cost_bp_one_way / 2) · Σ_L |r_L|` reproduces the two-leg flat fee
  exactly — which is what lets a *three*-leg package be priced at all.

The two numbers reported are the ones the rest of this package certifies with:
terminal gap and correlation of daily changes. A composition can hit the
terminal level by luck while taking a different path, and only the daily
correlation refuses to let that pass.

In [7]:
CERT_COMPOSE = fns.certify_dv01_neutral(LEGS, DATA, cfg=S1)
print(CERT_COMPOSE[["structure", "engine_terminal_usd", "composed_terminal_usd",
                    "terminal_gap_usd", "terminal_gap_pct", "corr_daily_changes",
                    "max_abs_daily_gap_usd", "n_marks_common"]].to_string(index=False))
print(f"\nworst terminal gap  {CERT_COMPOSE['terminal_gap_pct'].abs().max():.3e} %")
print(f"worst daily corr    {CERT_COMPOSE['corr_daily_changes'].min():.8f}")
print(f"worst daily gap     ${CERT_COMPOSE['max_abs_daily_gap_usd'].max():.2e}")
assert CERT_COMPOSE["terminal_gap_pct"].abs().max() < 1e-9
assert CERT_COMPOSE["corr_daily_changes"].min() > 1 - 1e-12
print("\nMachine precision on all four, on the level AND on the path. The fee")
print("model reproduces $100,000 per closed cohort round trip exactly: 79 closed")
print("x $100k = $7.9m, which is the whole difference between the fee-free")
print("composition and the engine's net curve.")

      structure  engine_terminal_usd  composed_terminal_usd  terminal_gap_usd  terminal_gap_pct  corr_daily_changes  max_abs_daily_gap_usd  n_marks_common
         5Y/30Y         3.749797e+07           3.749797e+07      1.639128e-07      4.371244e-13                 1.0           1.639128e-07            1907
        30Y/50Y         3.414660e+07           3.414660e+07     -2.235174e-08     -6.545818e-14                 1.0           6.705523e-08            1907
  20Yx5Y/25Yx5Y         4.940914e+07           4.940914e+07     -7.450581e-09     -1.507936e-14                 1.0           4.470348e-08            1907
10Yx10Y/20Yx10Y         4.572064e+07           4.572064e+07     -8.940697e-08     -1.955506e-13                 1.0           1.043081e-07            1907

worst terminal gap  4.371e-13 %
worst daily corr    1.00000000
worst daily gap     $1.64e-07

Machine precision on all four, on the level AND on the path. The fee
model reproduces $100,000 per closed cohort round trip exactl

## 4. The weights — walk-forward, and the look-ahead audit stated explicitly

**There are two PCAs in this study and they do different jobs.**

The **weight** basis is walk-forward: re-fitted at every cohort entry on data
strictly before it (expanding, `min_fit_days = 250`), because a weight is a
*decision* and a decision cannot use tomorrow's covariance. Fitting on the full
2019–2026 sample is right for the attribution report — that is a description of
what happened — and would be look-ahead here, because a 2019 cohort would be
sized with 2026's covariance.

The **measurement** basis is the committed full-sample `FactorModel` above.
Every exposure table, every variance share and the re-run attribution use it and
only it. Re-fitting a different PCA to score the new books would make the
*"did the level share move toward zero?"* comparison a comparison of two bases
instead of two sizings.

### The look-ahead audit, item by item

| quantity | what it may see | leak? |
|---|---|---|
| `pc1_neutral` / `pc12_neutral` weights at cohort *k* | curve days `< entry_k` | none |
| eigenvector labelling | previous fit only (`_match_pcs` + `classify_pcs`) | none |
| `slope_beta_hedged` beta at rebalance *m* | its own gross P&L, days `< entry_m` | none |
| the **factor** that beta is a beta *to* | `V` fitted at `entry_m` | none — **fixed here** |
| greeks / carry | the cohort's own entry date | none |
| the trade set (91 cohorts, 1-year holds) | fixed, identical for every sizing | none |
| `residual_exposure`, attribution shares | full sample, by design | **measurement only** |

The fourth row is the one the draft got wrong. The beta was trailing, but it was
a beta to a *definition of slope* estimated at the end of the sample. Truncating
the window does not fix that; `walk_forward_scores` does.

### Two things that are NOT dropped, on purpose

Cohort 0 enters 2019-02-01 and the panel starts 2019-01-02, so it has ~20 daily
changes and nothing can be done about it — `FactorConfig.start` is the first day
the full 2Y..50Y grid prices locally. Eleven cohorts sit under `min_fit_days`.
They are **flagged, not dropped**: dropping them would change the trade set and
stop this being a sizing comparison at all. Section 9 re-scores every book
without them as a robustness row.

In [8]:
LBD, WF_DIAG, VBD = fns.walk_forward_loadings(
    PANEL, FCFG, ENTRIES, fns.LEG_UNIVERSE,
    min_fit_days=CFG.min_fit_days, hard_floor=CFG.hard_fit_floor)
print(f"{len(LBD)} walk-forward fits, {WF_DIAG['n_fit_days'].min()} .. "
      f"{WF_DIAG['n_fit_days'].max()} days of history")
print(f"short fits (< {CFG.min_fit_days} days): {int(WF_DIAG['short_fit'].sum())}")
print(f"non-canonical PC labels: {int((~WF_DIAG['label_ok']).sum())}")
print()
print(WF_DIAG.head(6).to_string(index=False))
print("...")
print(WF_DIAG.tail(3).to_string(index=False))

91 walk-forward fits, 21 .. 1896 days of history
short fits (< 250 days): 11
non-canonical PC labels: 0

      date  n_fit_days  short_fit label_PC1 label_PC2 label_PC3   ev_PC1   ev_PC2   ev_PC3  label_ok
2019-02-01          21       True     level     slope curvature 0.951796 0.043859 0.004345      True
2019-03-01          40       True     level     slope curvature 0.951569 0.043933 0.004498      True
2019-04-01          61       True     level     slope curvature 0.957434 0.037910 0.004655      True
2019-05-01          82       True     level     slope curvature 0.959796 0.035680 0.004524      True
2019-06-03         104       True     level     slope curvature 0.962906 0.032196 0.004898      True
2019-07-01         124       True     level     slope curvature 0.949821 0.044930 0.005249      True
...
      date  n_fit_days  short_fit label_PC1 label_PC2 label_PC3   ev_PC1   ev_PC2   ev_PC3  label_ok
2026-06-01        1853      False     level     slope curvature 0.885626 0.106994 0

In [9]:
W = fns.cohort_weights(SCHED, fa.STRAT1_STRUCTURES, LBD, hedge_leg=CFG.hedge_leg,
                       dv01=DV01, neutralize=CFG.neutralize,
                       sizings=fns.STATIC_SIZINGS)
_cached = pd.read_parquet(DATA / "fns_cohort_weights.parquet")
_k = ["structure", "sizing", "cohort", "leg"]
_gap = float(np.abs(W.sort_values(_k)["dv01"].to_numpy()
                    - _cached.sort_values(_k)["dv01"].to_numpy()).max())
print(f"walk-forward weights reproduce the cached table to {_gap:.3e} $ of DV01")
assert _gap < 1e-6, "the weights the engine certified are not the weights composed"
print("This matters: the genuine engine run in section 11 was fed the CACHED")
print("table, so a drift between the two would certify a different book.")
print()
WSUM = fns.weights_summary(W, WF_DIAG)
print(WSUM.round(1).to_string(index=False))

walk-forward weights reproduce the cached table to 0.000e+00 $ of DV01
This matters: the genuine engine run in section 11 was fed the CACHED
table, so a drift between the two would certify a different book.

      structure       sizing  n_cohorts  gross_dv01_mean  net_dv01_mean  n_short_fit  n_label_bad  dv01_30Y_mean  dv01_30Y_sd  dv01_5Y_mean  dv01_5Y_sd  dv01_10Y_mean  dv01_10Y_sd  dv01_50Y_mean  dv01_50Y_sd  dv01_20Yx5Y_mean  dv01_20Yx5Y_sd  dv01_25Yx5Y_mean  dv01_25Yx5Y_sd  dv01_10Yx10Y_mean  dv01_10Yx10Y_sd  dv01_20Yx10Y_mean  dv01_20Yx10Y_sd
         5Y/30Y dv01_neutral         91         200000.0            0.0           11            0      -100000.0          0.0      100000.0         0.0            NaN          NaN            NaN          NaN               NaN             NaN               NaN             NaN                NaN              NaN                NaN              NaN
         5Y/30Y  pc1_neutral         91         200701.9          701.9           11            

`gross_dv01_mean` is the whole story in one column. `dv01_neutral` is $200,000
by construction. `pc1_neutral` is *near* it — the two legs are close in PC1
space — but never equal, and the deviation is the trade. `pc12_neutral` on
5Y/30Y is **$364,300**, 1.8x the incumbent's gross risk, because the 10Y hedge
leg needed to zero a slope exposure of $56,249 per unit is large. That extra
gross DV01 is what the cost model has to charge for, and it is exactly what the
flat per-package fee could not express.

## 5. Did the hedge remove the factor? — residual exposure

This is the half of the test the P&L cannot answer. A hedge that moves the P&L
to zero has not necessarily removed the factor, and a hedge that leaves the P&L
alone has not necessarily failed to. **If the shares do not move, the hedge is
not doing what it claims regardless of what the P&L does.**

Weights are walk-forward; exposures are measured on the shared full-sample
basis. Measuring a walk-forward-weighted package on its *own* walk-forward basis
would report `f_PC1 = 0` by construction and prove nothing at all. The question
is whether a weight chosen on 2019–2022 information is still level-neutral when
scored on the ruler everything else in this package is scored on.

In [10]:
RESID = fns.residual_exposure_table(W, FM)
print(RESID[["structure", "sizing", "n_legs", "gross_dv01_mean",
             "abs_f_level_mean", "abs_f_slope_mean", "abs_f_curv_mean",
             "var_level", "var_slope", "var_curv"]].round(4).to_string(index=False))

      structure       sizing  n_legs  gross_dv01_mean  abs_f_level_mean  abs_f_slope_mean  abs_f_curv_mean  var_level  var_slope  var_curv
         5Y/30Y dv01_neutral       2      200000.0000         4712.8966        56249.3254       55899.6476     0.0514     0.8886    0.0600
         5Y/30Y  pc1_neutral       2      200701.9207         5236.7104        56446.9244       56159.2601     0.0889     0.8540    0.0571
         5Y/30Y pc12_neutral       3      364300.3158          727.8096          830.4870       50112.7330     0.0385     0.0081    0.9534
        30Y/50Y dv01_neutral       2      200000.0000         1041.8373         2462.9212        9111.2515     0.4327     0.2932    0.2741
        30Y/50Y  pc1_neutral       2      198131.0157          564.2299         2988.0711        9464.7444     0.2140     0.4725    0.3135
        30Y/50Y pc12_neutral       3      211717.2711          528.2174         1082.6234        5794.8418     0.2936     0.1556    0.5508
  20Yx5Y/25Yx5Y dv01_neutra

In [11]:
_piv = RESID.pivot(index="structure", columns="sizing", values="var_level")
_piv = _piv[[c for c in ("dv01_neutral", "pc1_neutral", "pc12_neutral") if c in _piv]]
_piv["pc1_reduction"] = 1.0 - _piv["pc1_neutral"] / _piv["dv01_neutral"]
_piv["pc12_reduction"] = 1.0 - _piv["pc12_neutral"] / _piv["dv01_neutral"]
print("LEVEL share of the package's own factor variance:")
print(_piv.round(4).to_string())
print()
_pivs = RESID.pivot(index="structure", columns="sizing", values="var_slope")
_pivs = _pivs[[c for c in ("dv01_neutral", "pc1_neutral", "pc12_neutral") if c in _pivs]]
_pivs["pc12_reduction"] = 1.0 - _pivs["pc12_neutral"] / _pivs["dv01_neutral"]
print("SLOPE share of the package's own factor variance:")
print(_pivs.round(4).to_string())

LEVEL share of the package's own factor variance:
sizing           dv01_neutral  pc1_neutral  pc12_neutral  pc1_reduction  pc12_reduction
structure                                                                              
10Yx10Y/20Yx10Y        0.6134       0.0833        0.0859         0.8641          0.8600
20Yx5Y/25Yx5Y          0.6898       0.1017        0.1174         0.8526          0.8298
30Y/50Y                0.4327       0.2140        0.2936         0.5054          0.3214
5Y/30Y                 0.0514       0.0889        0.0385        -0.7278          0.2510

SLOPE share of the package's own factor variance:
sizing           dv01_neutral  pc1_neutral  pc12_neutral  pc12_reduction
structure                                                               
10Yx10Y/20Yx10Y        0.0021       0.2420        0.0677        -31.2194
20Yx5Y/25Yx5Y          0.1466       0.6027        0.1452          0.0096
30Y/50Y                0.2932       0.4725        0.1556          0.4694
5Y/30Y

**Read this table before any P&L.** On the three tight pairs `pc1_neutral` cuts
the level share by 51%, 85% and 86%; `pc12_neutral` cuts the slope share on
5Y/30Y by 99.1%. The hedges are doing what they claim.

The two rows that do **not** behave are the finding, not a defect:

* **5Y/30Y `pc1_neutral` makes level *worse*** (0.0514 → 0.0889). Level was never
  that book's problem — 5.1% of its variance — and the walk-forward PC1 shape
  differs enough from the full-sample one that "zeroing level on 2020 data"
  leaves more level on the 2019–2026 ruler than doing nothing. Section 9 prices
  that gap and it is large.
* **Every `pc1_neutral` row raises the SLOPE share.** That is arithmetic, not
  leakage: total factor variance shrinks, so what remains is a bigger fraction
  of a smaller number. The dollar exposure `abs_f_slope_mean` is the column to
  read against, and on 30Y/50Y it rises from $2,463 to $2,988 — a real, modest
  increase, the price of a two-leg trade with only one constraint to spend.

Note also that `pc12_neutral` does not drive the analytic level and slope
exposures to *exactly* zero here, and should not: the weights were solved on
each cohort's own walk-forward basis and are being scored on the full-sample
one. The residue (`var_level` 0.04–0.29 against 0.05–0.69 unhedged) is the
honest cost of not knowing the covariance in advance, and section 10 confirms
it on realised P&L rather than on the solve.

## 6. The books

Every sizing trades the **same 91 monthly cohorts on the same 1-year holds**.
Only the leg weights differ. Costs are charged per LEG:

```
fee_round_trip = 2 · (cost_bp_one_way / 2) · Σ_L |r_L|
```

At `r = (+100k, −100k)` that is `2 · 0.25 · 200,000 = $100,000`, exactly the flat
fee `build_backtest` charges — so this is not a new cost assumption, it is the
same one written so a three-leg package can pay for three legs. Carry is not
charged separately and must not be: the engine holds real swaps for a year and
marks them, so realised P&L already contains it. It is reported as a
decomposition column (`mean_carry_bp`), never as a second subtraction.

In [12]:
EQ, BK, CARRY_USD = {}, {}, {}
for (_lab, _sz), _g in W.groupby(["structure", "sizing"], sort=False):
    EQ[(_lab, _sz)], BK[(_lab, _sz)] = fns.compose_book(LEGS, _g, cfg=S1)
    _ck = {}
    for _k2, _gk in _g.groupby("cohort"):
        _d = pd.Timestamp(_gk["entry"].iloc[0])
        _cb = fns.leg_carry_bp(GREEKS, on=_d)
        _ck[_d] = float(sum(float(_r) * float(_cb.get(_l, np.nan))
                            for _l, _r in zip(_gk["leg"], _gk["dv01"])))
    CARRY_USD[(_lab, _sz)] = pd.Series(_ck).sort_index()
print(f"composed {len(EQ)} static books")

composed 12 static books


## 7. `slope_beta_hedged` — the empirical variant, and its honest cost

The incumbent package plus a DV01-neutral 5s30s overlay, sized by the beta of
the book's own daily gross P&L on `dPC2`, estimated on a **trailing** 252-day
window and rebalanced monthly. Both the beta and the factor definition are
walk-forward (section 4).

The beta is fitted on **gross** P&L, not net: the cost model books a $100k step
on the dozen days a cohort unwinds, and a step function is not a factor
exposure — including it would bias the beta by whatever the curve happened to do
on unwind days.

**Cost is a full round trip every month.** The overlay as constructed opens a
fresh at-market spread each month rather than adjusting a held one, so it pays
to get out and back in. That is the expensive reading and it is the headline;
`overlay_cost_sensitivity` reports the cheaper delta-traded bound beside it so
the reader can see the churn rather than take one number on trust.

In [13]:
SCORES_AT = {d: fns.walk_forward_scores(PANEL, FCFG, VBD[d], d, window=CFG.beta_window)
             for d in ENTRIES}
OVERLAY = {}
for _lab, _f, _b in fa.STRAT1_STRUCTURES:
    _wsub = W[(W["structure"] == _lab) & (W["sizing"] == "dv01_neutral")]
    _gross, _ = fns.compose_book(LEGS, _wsub, cfg=S1, cost_bp_one_way=0.0)
    _betas = fns.slope_beta_path(_gross, NLIVE, SCORES_AT, ENTRIES,
                                 window=CFG.beta_window)
    _sched = SCHED.assign(structure=_lab)
    _oeq, _obk, _hp = fns.compose_overlay(LEGS, _sched, _betas, LBD, cfg=S1)
    EQ[(_lab, "slope_beta_hedged")], BK[(_lab, "slope_beta_hedged")] = fns.add_overlay(
        EQ[(_lab, "dv01_neutral")], BK[(_lab, "dv01_neutral")], _oeq, _obk, cfg=S1)
    CARRY_USD[(_lab, "slope_beta_hedged")] = CARRY_USD[(_lab, "dv01_neutral")]
    OVERLAY[_lab] = {"betas": _betas, "book": _obk, "hedge_path": _hp,
                     "cost": fns.overlay_cost_sensitivity(_obk, _hp, _sched, cfg=S1)}

OVL_SUMMARY = pd.DataFrame([{
    "structure": _lab,
    "beta_mean_usd_per_pc2": float(_o["betas"]["beta_usd_per_pc2"].mean()),
    "beta_t_median": float(_o["betas"]["t_stat"].median()),
    "beta_r2_median": float(_o["betas"]["r2"].median()),
    "n_short_window": int(_o["betas"]["short_window"].sum()),
    "hedge_dv01_per_leg_mean": float(_o["hedge_path"]["hedge_dv01_per_leg"].mean()),
    "hedge_vs_package": abs(float(_o["hedge_path"]["hedge_dv01_per_leg"].mean())) / DV01,
    "overlay_gross_usd": float(_o["book"]["overlay_gross_usd"].sum()),
    "overlay_cost_usd": float(_o["book"]["overlay_cost_usd"].sum()),
    "overlay_net_usd": float(_o["book"]["overlay_net_usd"].sum()),
    "cost_delta_traded_usd": float(_o["cost"].loc[1, "total_cost_usd"]),
    "churn_ratio": float(_o["cost"].loc[2, "total_cost_usd"]),
} for _lab, _o in OVERLAY.items()])
print(OVL_SUMMARY.round(3).to_string(index=False))

      structure  beta_mean_usd_per_pc2  beta_t_median  beta_r2_median  n_short_window  hedge_dv01_per_leg_mean  hedge_vs_package  overlay_gross_usd  overlay_cost_usd  overlay_net_usd  cost_delta_traded_usd  churn_ratio
         5Y/30Y              63316.313         52.096           0.917              44              -108267.984             1.083      -3.521958e+07      1.165408e+08    -1.517603e+08            4639423.378       25.120
        30Y/50Y               4965.597          4.565           0.086              44                -8521.570             0.085       3.740482e+07      1.054499e+07     2.685983e+07            1031866.572       10.219
  20Yx5Y/25Yx5Y               3075.603          2.180           0.029              44                -5219.069             0.052       2.570527e+07      7.060664e+06     1.864461e+07             774286.383        9.119
10Yx10Y/20Yx10Y               1664.344          0.392           0.010              44                -2866.997             0

In [14]:
_rec = []
for _lab, _o in OVERLAY.items():
    _oeq = EQ[(_lab, "slope_beta_hedged")] - EQ[(_lab, "dv01_neutral")]
    _rec.append({"structure": _lab,
                 "overlay_terminal_equity": float(_oeq.iloc[-1]),
                 "sum_per_cohort_net": float(_o["book"]["overlay_net_usd"].sum()),
                 "gap": float(_oeq.iloc[-1] - _o["book"]["overlay_net_usd"].sum()),
                 "n_live_cohorts": int((~_o["book"]["closed"]).sum()),
                 "one_month_fee": float(2.0 * fns.leg_cost_bp_one_way(S1) * 2
                                        * abs(_o["hedge_path"]["hedge_dv01_per_leg"]
                                              .iloc[-1]))})
OVL_RECON = pd.DataFrame(_rec)
print("Overlay reconciliation -- equity path against the per-cohort book:")
print(OVL_RECON.round(2).to_string(index=False))
print("""
The residual is the twelve STILL-LIVE cohorts' final monthly fee. The equity
curve is a mark-to-market and does not charge an exit cost on a position that is
still open -- the same convention `compose_book` uses -- while the per-cohort
table books it. It touches no `net_bp`, because `book_stats` reads closed
cohorts only and a closed cohort's last overlay segment always ends before the
last mark. The reported `overlay_cost_usd` is therefore the CONSERVATIVE side of
a 0.4% bookkeeping difference, which is the side to be on.""")

Overlay reconciliation -- equity path against the per-cohort book:
      structure  overlay_terminal_equity  sum_per_cohort_net        gap  n_live_cohorts  one_month_fee
         5Y/30Y            -1.505233e+08       -1.517603e+08 1237011.07              12      103084.26
        30Y/50Y             2.696142e+07        2.685983e+07  101586.33              12        8465.53
  20Yx5Y/25Yx5Y             1.868106e+07        1.864461e+07   36448.30              12        3037.36
10Yx10Y/20Yx10Y             4.412505e+07        4.404632e+07   78734.64              12        6561.22

The residual is the twelve STILL-LIVE cohorts' final monthly fee. The equity
curve is a mark-to-market and does not charge an exit cost on a position that is
still open -- the same convention `compose_book` uses -- while the per-cohort
table books it. It touches no `net_bp`, because `book_stats` reads closed
cohorts only and a closed cohort's last overlay segment always ends before the
last mark. The reported `ove

**`hedge_vs_package` on 5Y/30Y is 1.08.** The slope hedge that neutralises that
book is *bigger than the book*. That is not a defect of the 5s30s choice: a
package whose slope exposure is $56,249 per unit of PC2 has no slope hedge that
is not essentially its own reverse, and the overlay costs **$116.5m** against a
$100k-DV01 package — 1163 bp of cost, four times its own gross P&L. The three
tight pairs need 2.9%–8.5% of package size and cost an order of magnitude less.
This is the same finding as the attribution's, arrived at from the cost side.

**Where the tight pairs' overlay P&L comes from, stated plainly.** The overlay
supplies **28% (20Yx5Y/25Yx5Y), 43% (30Y/50Y) and 47% (10Yx10Y/20Yx10Y)** of the
hedged book's total. That is not extra convexity; it is the *absence of a slope
loss*. The attribution shows these flatteners were losing on slope
(`share_slope` = −149% on 5Y/30Y, −10% on 30Y/50Y), so a position that cancels
slope adds back what slope took away. Its magnitude is a property of what slope
did in 2019–2026 and should not be extrapolated.

**And `beta_t_median` is the column that disciplines this book.** It reads
**52.1** on 5Y/30Y, **4.57** on 30Y/50Y, **2.18** on 20Yx5Y/25Yx5Y and
**0.39** on 10Yx10Y/20Yx10Y, with median R² of 0.917, 0.086, 0.029 and 0.010.
Only 5Y/30Y has a slope beta that is genuinely identified — which is precisely
what the attribution said, since that is the only structure with meaningful
slope variance (88.9% against 0.2% for 10Yx10Y/20Yx10Y).

So **the 10Yx10Y/20Yx10Y overlay is fitting noise.** A `t` of 0.39 on the
hedge ratio means there is no slope exposure to hedge, and the $44.0m the
overlay earned there is an unmotivated short-slope position that happened to
pay. It is reported at full size because suppressing it would be choosing the
result, but it must not be read as a hedge working. Section 13 carries that
caveat into the scoreboard.

`n_short_window` of 44 says half the rebalances ran on less than a full 252-day
window, which is unavoidable — the alternative is leaving the first years
unhedged and comparing two different trade sets.

## 8. The sizing-by-structure table

Everything net of its own hedge cost, with a gross column beside it so the
reader can see what the hedge took.

In [15]:
_rows = []
for (_lab, _sz), _eq in EQ.items():
    _bk = BK[(_lab, _sz)]
    _cl = _bk[_bk["closed"].to_numpy(bool)]
    _st = fns.book_stats(_eq, _bk, cfg=S1,
                         carry_bp=float(CARRY_USD[(_lab, _sz)].mean() / DV01),
                         business_days_per_year=CFG.business_days_per_year)
    _st.update({
        "structure": _lab, "sizing": _sz,
        "gross_bp_total": float(_cl["gross_bp"].sum()),
        "cost_bp_total": float(_cl["cost_bp"].sum()),
        "gross_dv01_mean": float(_cl["gross_dv01"].mean()),
        "breakeven_cost_bp_leg": fns.breakeven_cost_bp(_bk, cfg=S1),
    })
    _rows.append(_st)
STATS = pd.DataFrame(_rows).set_index(["structure", "sizing"]).sort_index()
HEAD = ["n_closed", "gross_bp_total", "cost_bp_total", "total_net_bp", "mean_net_bp",
        "hit_rate", "sharpe_per_trade", "t_stat_overlap_adj", "mtm_final_bp",
        "mtm_sharpe_ann", "mtm_max_dd_bp", "mean_carry_bp", "breakeven_cost_bp_leg"]
print("WALK-FORWARD weights. Every row net of ITS OWN hedge cost.")
print(STATS[HEAD].round(4).to_string())

WALK-FORWARD weights. Every row net of ITS OWN hedge cost.
                                   n_closed  gross_bp_total  cost_bp_total  total_net_bp  mean_net_bp  hit_rate  sharpe_per_trade  t_stat_overlap_adj  mtm_final_bp  mtm_sharpe_ann  mtm_max_dd_bp  mean_carry_bp  breakeven_cost_bp_leg
structure       sizing                                                                                                                                                                                                                  
10Yx10Y/20Yx10Y dv01_neutral             79        579.0958        79.0000      500.0958       6.3303    0.6203            0.2684              0.7351      457.2064          0.2209      -860.2164        -0.5299                 1.8326
                pc12_neutral             79        417.6498        80.7484      336.9014       4.2646    0.5190            0.1957              0.5360      261.9184          0.1407      -867.6910        -0.3445                 1.2931
         

In [16]:
_net = STATS["total_net_bp"].unstack("sizing")
_net = _net[[c for c in fns.SIZINGS if c in _net.columns]]
_rel = _net.div(_net["dv01_neutral"], axis=0)
print("total net bp, closed cohorts:")
print(_net.round(1).to_string())
print("\nas a multiple of the incumbent:")
print(_rel.round(3).to_string())

total net bp, closed cohorts:
sizing           dv01_neutral  pc1_neutral  pc12_neutral  slope_beta_hedged
structure                                                                  
10Yx10Y/20Yx10Y         500.1        334.8         336.9              931.7
20Yx5Y/25Yx5Y           479.6        426.3         400.0              663.0
30Y/50Y                 341.0        300.7         342.1              621.9
5Y/30Y                  268.8       1407.8        -739.8            -1026.6

as a multiple of the incumbent:
sizing           dv01_neutral  pc1_neutral  pc12_neutral  slope_beta_hedged
structure                                                                  
10Yx10Y/20Yx10Y           1.0        0.670         0.674              1.863
20Yx5Y/25Yx5Y             1.0        0.889         0.834              1.382
30Y/50Y                   1.0        0.882         1.003              1.824
5Y/30Y                    1.0        5.237        -2.752             -3.819


**The decisive read.** Neutralising the structure's own dominant factor:

* **survives** on 30Y/50Y (341 → 301 bp, −12%), 20Yx5Y/25Yx5Y (480 → 426 bp,
  −11%) and 10Yx10Y/20Yx10Y (500 → 335 bp, −33%). All three stay positive, all
  three keep hit rates of 0.53–0.73, and section 10 shows the convexity term
  intact at `t = 4.1–4.9`. The incumbent sizing was *not* diluting a bigger
  convexity trade — the level exposure it carried was mildly **profitable** over
  this sample, which is why removing it costs 11–33% rather than adding. What
  removing it buys is that the remaining P&L is the thing the book claims to
  trade.
* **collapses** on 5Y/30Y under `pc12_neutral`: **+269 → −740 bp**, hit rate
  0.53 → 0.20, Sharpe/trade +0.065 → −0.583. Remove the slope and there is
  nothing underneath.

`slope_beta_hedged` reads *better* than the incumbent on the three tight pairs
(1.38x–1.86x). Section 7 and section 13 say why that is a risk reduction rather
than a new earner. `pc1_neutral` on 5Y/30Y reads **+1408 bp** and must not be
banked — see section 9.

**Break-even cost** is generalised here and had to be. The house
`breakeven_cost_bp` divides by `2 · n_closed` because every one of its packages
is the same two legs at the same size; that is exactly what stops working when
`pc12_neutral` trades three legs and `pc1_neutral` trades two of unequal size.
The denominator used is the gross DV01 actually traded, so the number is
comparable across sizings: **1.21–1.63 bp per leg** on the three surviving
headline books against a charged 0.25 bp — 4.8x to 6.5x headroom. The one
negative entry is `pc12_neutral` on 5Y/30Y (−1.04 bp): that book loses money
gross, so no cost level rescues it, and the number says so rather than being
clipped to zero.

In [17]:
COSTS = {}
for (_lab, _sz), _g in W.groupby(["structure", "sizing"], sort=False):
    COSTS[(_lab, _sz)] = fns.cost_sensitivity(LEGS, _g, cfg=S1,
                                              multipliers=CFG.cost_multipliers)
_cs = pd.concat({k: v.set_index("cost_multiple")["total_net_bp"]
                 for k, v in COSTS.items()}, axis=1).T
_cs.index.names = ["structure", "sizing"]
print("total net bp by cost multiple (1.0 = the committed 0.5 bp one way):")
print(_cs.sort_index().round(1).to_string())
print("\nThe 0.0 column is the hedge priced honestly: the gap between it and the")
print("1.0 column is exactly what the extra legs cost to trade.")

total net bp by cost multiple (1.0 = the committed 0.5 bp one way):
cost_multiple                    0.0     0.5     1.0     2.0
structure       sizing                                      
10Yx10Y/20Yx10Y dv01_neutral   579.1   539.6   500.1   421.1
                pc12_neutral   417.6   377.3   336.9   256.2
                pc1_neutral    411.1   373.0   334.8   258.6
20Yx5Y/25Yx5Y   dv01_neutral   558.6   519.1   479.6   400.6
                pc12_neutral   484.5   442.2   400.0   315.5
                pc1_neutral    503.4   464.8   426.3   349.2
30Y/50Y         dv01_neutral   420.0   380.5   341.0   262.0
                pc12_neutral   425.2   383.6   342.1   259.0
                pc1_neutral    379.1   339.9   300.7   222.4
5Y/30Y          dv01_neutral   347.8   308.3   268.8   189.8
                pc12_neutral  -595.9  -667.9  -739.8  -883.6
                pc1_neutral   1487.9  1447.9  1407.8  1327.7

The 0.0 column is the hedge priced honestly: the gap between it and the
1.0 c

## 9. Walk-forward against full-sample — the price of not knowing

The full-sample variant is computed for free and reported beside the headline.
If the two agree the look-ahead was immaterial and the comparison is
like-for-like; if they do not, **the walk-forward one is the answer and the gap
is the cost of not knowing the covariance in advance.** Measured, not asserted.

In [18]:
L_FULL = fns.full_sample_loadings(PANEL, FCFG, fns.LEG_UNIVERSE)
W_FULL = fns.cohort_weights(SCHED, fa.STRAT1_STRUCTURES,
                            {pd.Timestamp(_e): L_FULL for _e in ENTRIES},
                            hedge_leg=CFG.hedge_leg, dv01=DV01,
                            neutralize=CFG.neutralize, sizings=fns.STATIC_SIZINGS)
_rows = []
for (_lab, _sz), _g in W_FULL.groupby(["structure", "sizing"], sort=False):
    _eq, _bk = fns.compose_book(LEGS, _g, cfg=S1)
    _cl = _bk[_bk["closed"].to_numpy(bool)]
    _p = _cl["net_bp"].to_numpy(float)
    _rows.append({"structure": _lab, "sizing": _sz,
                  "full_sample_net_bp": float(_p.sum()),
                  "full_sample_sharpe": float(_p.mean() / _p.std(ddof=1))})
LOOKAHEAD = pd.DataFrame(_rows).set_index(["structure", "sizing"]).join(
    STATS[["total_net_bp", "sharpe_per_trade"]]).rename(
    columns={"total_net_bp": "walk_forward_net_bp",
             "sharpe_per_trade": "walk_forward_sharpe"})
LOOKAHEAD["net_bp_gap"] = LOOKAHEAD["walk_forward_net_bp"] - LOOKAHEAD["full_sample_net_bp"]
LOOKAHEAD["abs_gap_vs_wf"] = (LOOKAHEAD["net_bp_gap"].abs()
                              / LOOKAHEAD["walk_forward_net_bp"].abs())
print(LOOKAHEAD[["full_sample_net_bp", "walk_forward_net_bp", "net_bp_gap",
                 "abs_gap_vs_wf", "full_sample_sharpe",
                 "walk_forward_sharpe"]].round(3).to_string())

                              full_sample_net_bp  walk_forward_net_bp  net_bp_gap  abs_gap_vs_wf  full_sample_sharpe  walk_forward_sharpe
structure       sizing                                                                                                                   
5Y/30Y          dv01_neutral             268.803              268.803       0.000          0.000               0.065                0.065
                pc1_neutral              -66.081             1407.797    1473.878          1.047              -0.018                0.260
                pc12_neutral            -725.712             -739.789     -14.078          0.019              -0.536               -0.583
30Y/50Y         dv01_neutral             341.026              341.026       0.000          0.000               0.311                0.311
                pc1_neutral              267.473              300.746      33.273          0.111               0.228                0.266
                pc12_neutral      

**5Y/30Y `pc1_neutral` is where this matters and it disqualifies that number.**
The same sizing rule scores **−66 bp** with full-sample loadings and **+1408 bp**
walk-forward — a gap of 1474 bp, 105% of the walk-forward total. That is not a
result; it is the PCA moving. The mechanism is visible in the weights: the 5Y
leg of the walk-forward `pc1_neutral` ranges $70,557 to $134,875 (sd $18,902),
spiking during the 2020–2022 ZIRP window when the 5Y point barely moved with
level and PC1-neutrality therefore demanded a much bigger 5Y leg. The P&L that
produced is a bet on the estimator, not on convexity.

The three tight pairs are the opposite: full-sample and walk-forward agree to
3–20%, which is what "the look-ahead was immaterial" looks like when measured.

In [19]:
_wf5 = W[(W["structure"] == "5Y/30Y") & (W["sizing"] == "pc1_neutral")
         & (W["leg"] == "5Y")]["dv01"]
print(f"5Y/30Y pc1_neutral, walk-forward 5Y leg DV01: mean ${_wf5.mean():,.0f}  "
      f"sd ${_wf5.std(ddof=1):,.0f}  range ${_wf5.min():,.0f} .. ${_wf5.max():,.0f}")
print(f"full-sample equivalent: ${W_FULL[(W_FULL['structure'] == '5Y/30Y') & (W_FULL['sizing'] == 'pc1_neutral') & (W_FULL['leg'] == '5Y')]['dv01'].iloc[0]:,.0f}")

5Y/30Y pc1_neutral, walk-forward 5Y leg DV01: mean $100,702  sd $18,902  range $70,557 .. $134,875
full-sample equivalent: $85,702


### Robustness: the eleven thin-fit cohorts

Dropping them would change the trade set, so the headline keeps them. Re-scored
without them, here is what moves.

In [20]:
_thin = set(WF_DIAG.loc[WF_DIAG["short_fit"], "date"].astype("datetime64[ns]"))
_rows = []
for (_lab, _sz), _bk in BK.items():
    _cl = _bk[_bk["closed"].to_numpy(bool)]
    _keep = _cl[~pd.to_datetime(_cl["entry"]).isin(_thin)]
    _p, _q = _cl["net_bp"].to_numpy(float), _keep["net_bp"].to_numpy(float)
    _rows.append({"structure": _lab, "sizing": _sz,
                  "n_all": len(_p), "net_bp_all": float(_p.sum()),
                  "n_mature": len(_q), "net_bp_mature_only": float(_q.sum()),
                  "sharpe_all": float(_p.mean() / _p.std(ddof=1)),
                  "sharpe_mature": float(_q.mean() / _q.std(ddof=1))})
THIN = pd.DataFrame(_rows).set_index(["structure", "sizing"]).sort_index()
print(f"{len(_thin)} cohorts sized on a PCA shorter than {CFG.min_fit_days} days:")
print(THIN.round(3).to_string())

11 cohorts sized on a PCA shorter than 250 days:
                                   n_all  net_bp_all  n_mature  net_bp_mature_only  sharpe_all  sharpe_mature
structure       sizing                                                                                       
10Yx10Y/20Yx10Y dv01_neutral          79     500.096        68             344.483       0.268          0.205
                pc12_neutral          79     336.901        68             153.543       0.196          0.101
                pc1_neutral           79     334.837        68              90.312       0.176          0.056
                slope_beta_hedged     79     931.740        68             705.787       0.414          0.344
20Yx5Y/25Yx5Y   dv01_neutral          79     479.638        68             372.315       0.933          0.886
                pc12_neutral          79     399.975        68             258.944       0.929          0.887
                pc1_neutral           79     426.291        68         

## 10. The attribution, re-run ON THE NEW BOOKS

A successful hedge must move the neutralised factor toward zero *in the
realised P&L*, not only in the analytic exposure. The books are put back through
`factor_attribution.run_attribution` unchanged, on the same shared full-sample
basis, so a share here is the same object as a share in the attribution report
and the two tables can be read side by side.

Two readings, and they answer different questions:

* `share_*` — fraction of the realised P&L. The literal question. It can exceed
  100% or go negative whenever factors partly offset, which is normal.
* `incr_*` — incremental R². The **risk** reading, and the one that settles
  "did the hedge work". A book can carry a huge slope exposure that happened to
  earn nothing, and only the R² column shows it.

In [21]:
SERIES = {}
for (_lab, _sz), _eq in EQ.items():
    SERIES.update(fns.sizing_series(_lab, _sz, _eq, BK[(_lab, _sz)], CALENDAR,
                                    cfg=S1, carry_usd=CARRY_USD[(_lab, _sz)]))
ATTR_TRADE = fns.attribution_shares(FM, SERIES, FCFG, level="trade")
ATTR_DAILY = fns.attribution_shares(FM, SERIES, FCFG, level="daily")
_cols = ["strategy", "n_obs", "total_pnl", "r2", "share_level", "share_slope",
         "share_curvature", "share_convexity", "share_carry", "share_unexplained",
         "t_convexity", "denom_unstable"]
print("TRADE level (convexity regressor = squared TERMINAL move; the right one")
print("for an unhedged 1-year buy-and-hold cohort):")
print(ATTR_TRADE[_cols].round(3).to_string(index=False))

TRADE level (convexity regressor = squared TERMINAL move; the right one
for an unhedged 1-year buy-and-hold cohort):
                         strategy  n_obs     total_pnl    r2  share_level  share_slope  share_curvature  share_convexity  share_carry  share_unexplained  t_convexity  denom_unstable
      10Yx10Y/20Yx10Y pc1_neutral     79  3.348370e+07 0.791        0.461       -0.077           -0.410            3.162       -0.576             -1.559        4.244           False
     10Yx10Y/20Yx10Y pc12_neutral     79  3.369014e+07 0.747        0.529       -0.049           -0.322            2.901       -0.211             -1.849        4.587           False
              30Y/50Y pc1_neutral     79  3.007456e+07 0.808       -0.081       -0.124           -0.285            2.088       -0.440             -0.158        4.860           False
     10Yx10Y/20Yx10Y dv01_neutral     79  5.000958e+07 0.796        0.578       -0.011           -0.217            1.943       -0.497             -0.796   

In [22]:
_icols = ["strategy", "r2", "incr_level", "incr_slope", "incr_curvature",
          "incr_convexity", "incr_carry", "t_level", "t_slope", "t_convexity"]
print("DAILY level, INCREMENTAL R^2 -- the risk reading:")
print(ATTR_DAILY[_icols].round(4).to_string(index=False))

DAILY level, INCREMENTAL R^2 -- the risk reading:
                         strategy     r2  incr_level  incr_slope  incr_curvature  incr_convexity  incr_carry  t_level  t_slope  t_convexity
      10Yx10Y/20Yx10Y pc1_neutral 0.2385      0.0731      0.0307          0.1313          0.0028      0.0006   2.6650   1.7612       1.2645
     10Yx10Y/20Yx10Y pc12_neutral 0.1925      0.0788      0.0026          0.1082          0.0029      0.0000   2.8040   0.5429       1.1530
     10Yx10Y/20Yx10Y dv01_neutral 0.3102      0.2153      0.0012          0.0903          0.0027      0.0007   4.8971   0.4033       1.3627
10Yx10Y/20Yx10Y slope_beta_hedged 0.3121      0.2487      0.0002          0.0578          0.0036      0.0017   5.4042  -0.1162       1.6818
        20Yx5Y/25Yx5Y pc1_neutral 0.1850      0.0194      0.1005          0.0628          0.0022      0.0001   1.2919   2.9648       1.4053
              5Y/30Y dv01_neutral 0.9436      0.0099      0.8828          0.0504          0.0001      0.0004  

In [23]:
_d = ATTR_DAILY.copy()
_d["structure"] = _d["strategy"].str.rsplit(" ", n=1).str[0]
_d["sizing"] = _d["strategy"].str.rsplit(" ", n=1).str[1]
MOVE = pd.DataFrame({
    "incr_level": _d.pivot(index="structure", columns="sizing", values="incr_level").stack(),
    "incr_slope": _d.pivot(index="structure", columns="sizing", values="incr_slope").stack(),
}).unstack("sizing")
print("Did the neutralised factor's RISK share move toward zero? (daily incr R^2)")
print(MOVE.round(4).to_string())
print()
for _s in ("20Yx5Y/25Yx5Y", "10Yx10Y/20Yx10Y", "30Y/50Y"):
    _a = float(MOVE[("incr_level", "dv01_neutral")][_s])
    _b = float(MOVE[("incr_level", "pc1_neutral")][_s])
    print(f"{_s:18s} level  {_a:.4f} -> {_b:.4f}   ({100*(1-_b/_a) if _a else np.nan:+.0f}%)")
_a = float(MOVE[("incr_slope", "dv01_neutral")]["5Y/30Y"])
_b = float(MOVE[("incr_slope", "pc12_neutral")]["5Y/30Y"])
print(f"{'5Y/30Y':18s} slope  {_a:.4f} -> {_b:.4f}   ({100*(1-_b/_a):+.0f}%)")

Did the neutralised factor's RISK share move toward zero? (daily incr R^2)
                  incr_level                                              incr_slope                                           
sizing          dv01_neutral pc12_neutral pc1_neutral slope_beta_hedged dv01_neutral pc12_neutral pc1_neutral slope_beta_hedged
structure                                                                                                                      
10Yx10Y/20Yx10Y       0.2153       0.0788      0.0731            0.2487       0.0012       0.0026      0.0307            0.0002
20Yx5Y/25Yx5Y         0.1616       0.0139      0.0194            0.2029       0.0235       0.0016      0.1005            0.0018
30Y/50Y               0.0111       0.0006      0.0005            0.0270       0.1044       0.0617      0.1176            0.0030
5Y/30Y                0.0099       0.0286      0.0172            0.0213       0.8828       0.0117      0.8049            0.0015

20Yx5Y/25Yx5Y      level  0.

**The hedges work.** Level's incremental R² falls 88% on 20Yx5Y/25Yx5Y, 66% on
10Yx10Y/20Yx10Y and to zero on 30Y/50Y; slope's falls **99%** on 5Y/30Y. That is
measured on realised P&L, on the same ruler as the attribution report, and it is
the part of the answer that does not depend on a small sample.

**And the convexity term survives on the tight pairs.** At TRADE level — where
the convexity regressor is the squared terminal move, the right one for a
1-year buy-and-hold — `incr_convexity` on the three tight pairs' `pc1_neutral`
books is **0.317, 0.339, 0.360** at `t_convexity` = **4.1–4.9**, against the
incumbent's 0.321, 0.306, 0.348 at t = 3.5–4.9. Including `pc12_neutral` widens
the t range to 3.4–5.2 and leaves the incremental R² in the same band. The
hedge removed the factor and left the convexity exactly where it was, which is
the good outcome and the one the prior predicted.

**5Y/30Y is the counter-case, and it is instructive.** Its incumbent book has
`incr_slope = 0.677` and `incr_convexity = 0.004` at trade level (0.883 and
0.0001 daily) — the convexity term explains essentially *nothing*. Hedge the
slope out and `incr_convexity` rises to 0.315, not because convexity started
earning but because it is now the largest thing left, while the P&L goes to
−740 bp. The convexity was always there as an *exposure*; it was never the
earner.

## 11. Certification II — a genuine multi-leg engine run of the new weights

Section 3 certified the composition *arithmetic* against four stored two-leg
runs. It cannot certify the **new weights**, because those vary by cohort and
are not equal and opposite — the one thing the stored unit runs never exercise.
So `5Y/30Y × pc12_neutral`, the structure the whole premise rests on and the one
whose weights move most, is re-run as a genuine three-leg `QueryDrivenBacktest`
with 273 tagged leg-positions, fed the cached weight table verbatim.

`QueryDrivenBacktest.run()` **swallows exceptions**, so a dead run is an empty
`mtm_history` and never a traceback. The build script asserts on it; so does
this cell.

In [24]:
_safe = f"{fns.safe_leg(CFG.certify_structure)}_{CFG.certify_sizing}"
_ep = DATA / f"fns_certify_equity_{_safe}.parquet"
_cp = DATA / f"fns_certify_cohorts_{_safe}.parquet"
if _ep.exists() and _cp.exists():
    ENG = pd.read_parquet(_ep)["equity_usd"].astype(float)
    ENG.index = pd.to_datetime(ENG.index)
    ENG_COH = pd.read_parquet(_cp)
    assert len(ENG) > 0, "certify run produced no mtm_history"
    _composed = EQ[(CFG.certify_structure, CFG.certify_sizing)]
    # The engine run charges fee=0 on the unwind, exactly as the leg runs do, so
    # it is compared against the FEE-FREE composition. The fee model itself is
    # already certified exactly in section 3.
    _gw = W[(W["structure"] == CFG.certify_structure)
            & (W["sizing"] == CFG.certify_sizing)]
    _gross_composed, _ = fns.compose_book(LEGS, _gw, cfg=S1, cost_bp_one_way=0.0)
    CERT_ENGINE = fns.certify_engine(ENG, _gross_composed,
                                     label=CFG.certify_structure,
                                     sizing=CFG.certify_sizing, cfg=S1)
    print(json.dumps({k: (round(v, 10) if isinstance(v, float) else v)
                      for k, v in CERT_ENGINE.items()}, indent=1, default=str))
    print(f"\ncohorts closed in the engine run: {int(ENG_COH['closed'].sum())} / {len(ENG_COH)}")
    print(f"leg-positions opened: {int(ENG_COH['n_legs'].sum())}")
    assert abs(CERT_ENGINE["terminal_gap_pct"]) < 1e-6
    assert CERT_ENGINE["corr_daily_changes"] > 1 - 1e-9

    # A second, independent leg of the same certification: the engine's own
    # per-cohort REALISED P&L against the weighted sum of the per-leg realised
    # P&L. The equity comparison above tests the marks; this tests the unwind,
    # which is a different code path (`closed_positions_log`, not
    # `_position_value`) and the one that carries the fee convention.
    _by = ENG_COH.set_index("cohort")["gross_pnl_ccy"].astype(float)
    _mine = {}
    for _k3, _gk in _gw.groupby("cohort"):
        if not bool(_gk["closed"].iloc[0]):
            continue
        _t = 0.0
        for _, _r in _gk.iterrows():
            _cr = LEGS[_r["leg"]].cohorts
            _t += (float(_r["dv01"]) / fns.UNIT_DV01
                   * float(_cr.loc[_cr["cohort"] == _k3, "gross_pnl_ccy"].iloc[0]))
        _mine[int(_k3)] = _t
    _m = pd.Series(_mine)
    _e = _by.reindex(_m.index)
    print(f"per-cohort realised P&L: n={len(_m)}  "
          f"max abs err ${float((_m - _e).abs().max()):.6f}  "
          f"max rel {float(((_m - _e).abs() / _e.abs()).max()):.3e}  "
          f"corr {float(_m.corr(_e)):.10f}")
    assert float(((_m - _e).abs() / _e.abs()).max()) < 1e-9
    print("\nThe composed pc12_neutral book IS the engine's own three-leg run, on")
    print("the level, on the path AND on every individual unwind. Every other book")
    print("in this notebook is built by the same arithmetic from the same eight")
    print("mark matrices, so this certifies all sixteen.")
    print("\nNote the engine run carries 1908 marks to the composition's 1907: it")
    print("marked Good Friday 2019-04-19 because its 5Y leg priced that day. The")
    print("comparison is on the intersection, which is the grid section 2 fixed.")
else:
    CERT_ENGINE = {}
    raise FileNotFoundError(
        f"{_ep} missing -- run `_factor_neutral_build.py certify "
        f"'{CFG.certify_structure}' {CFG.certify_sizing}` first")

{
 "structure": "5Y/30Y",
 "n_marks_common": 1907,
 "engine_terminal_usd": -70097993.84689677,
 "composed_terminal_usd": -70097993.84689663,
 "engine_terminal_bp": -700.979938469,
 "composed_terminal_bp": -700.979938469,
 "terminal_gap_usd": -1.341e-07,
 "terminal_gap_pct": -0.0,
 "max_abs_daily_gap_usd": 2.98e-07,
 "corr_daily_changes": 1.0,
 "n_daily_changes": 1906,
 "sizing": "pc12_neutral"
}

cohorts closed in the engine run: 79 / 91
leg-positions opened: 273
per-cohort realised P&L: n=79  max abs err $0.000000  max rel 1.270e-13  corr 1.0000000000

The composed pc12_neutral book IS the engine's own three-leg run, on
the level, on the path AND on every individual unwind. Every other book
in this notebook is built by the same arithmetic from the same eight
mark matrices, so this certifies all sixteen.

Note the engine run carries 1908 marks to the composition's 1907: it
marked Good Friday 2019-04-19 because its 5Y leg priced that day. The
comparison is on the intersection, which is 

## 12. The analytic sweeps — exposures and greeks, deliberately NOT P&L

`HEDGE_LEG = 10Y` is pre-committed on stated structural grounds: it is the most
liquid point on the USD curve, it sits in the middle of the PCA grid so its
loading vector is genuinely independent of both long-end legs, and it gives a
well-conditioned solve for all four structures. The sweep below shows the whole
frontier so that choice is **checkable** — in exposures, conditioning, gamma and
carry, and **never in realised P&L**, because ranking nine hedge tenors on
returns and reporting the winner would be nine trials wearing one trial's
clothes.

In [25]:
FRONTIER = fns.hedge_tenor_frontier(FM, gamma=GAMMA, carry=CARRY)
print(FRONTIER[["structure", "hedge_leg", "ok", "cond", "dv01_hedge", "gross_dv01",
                "hedge_frac_of_gross", "var_curv_residual", "gamma_usd_per_bp2",
                "gamma_vs_dv01n", "carry_usd_1y"]].round(4).to_string(index=False))
print(f"\n10Y condition numbers: "
      f"{FRONTIER[FRONTIER['hedge_leg'] == '10Y']['cond'].round(2).tolist()}")
print("Best-conditioned admissible hedge per structure:")
print(FRONTIER[FRONTIER["ok"]].loc[
    FRONTIER[FRONTIER["ok"]].groupby("structure")["cond"].idxmin(),
    ["structure", "hedge_leg", "cond"]].to_string(index=False))

      structure hedge_leg   ok    cond   dv01_hedge  gross_dv01  hedge_frac_of_gross  var_curv_residual  gamma_usd_per_bp2  gamma_vs_dv01n  carry_usd_1y
         5Y/30Y        2Y True  5.4055 -171624.4455 502806.0374               0.3413                1.0           190.1795          0.8591  3818312.9039
         5Y/30Y        3Y True  7.2158 -252571.3141 673059.4275               0.3753                1.0           187.1115          0.8453  3197778.6481
         5Y/30Y        7Y True  6.0479  337866.1943 694556.2026               0.4864                1.0           164.5048          0.7431  1352106.9514
         5Y/30Y       10Y True  2.6537  176557.1114 366328.9011               0.4820                1.0           142.9181          0.6456   892053.1294
         5Y/30Y       15Y True  1.6387  126315.5480 259807.8932               0.4862                1.0           106.9498          0.4831   501448.9600
         5Y/30Y       20Y True  1.3486  111605.6232 226993.6617               0.49

In [26]:
SLOPE_TAB = fns.slope_instrument_table(FM)
print("Slope overlay candidates, ranked on curvature injected per unit of slope")
print("hedged. A slope hedge that carries curvature swaps one factor for another:")
print(SLOPE_TAB.round(4).to_string(index=False))

Slope overlay candidates, ranked on curvature injected per unit of slope
hedged. A slope hedge that carries curvature swaps one factor for another:
instrument              legs  f_level_per_unit  f_slope_per_unit  f_curv_per_unit  abs_curv_per_slope  abs_level_per_slope  selected
     5s10s  5Y+1.0 / 10Y-1.0            0.0020            0.2975          -0.0080              0.0268               0.0068     False
     5s30s  5Y+1.0 / 30Y-1.0            0.0471            0.5625          -0.5590              0.9938               0.0838      True
     2s10s  2Y+1.0 / 10Y-1.0           -0.0482            0.5589           0.9278              1.6600               0.0862     False
    10s20s 10Y+1.0 / 20Y-1.0            0.0290            0.1969          -0.3709              1.8835               0.1474     False
    10s30s 10Y+1.0 / 30Y-1.0            0.0451            0.2650          -0.5510              2.0795               0.1702     False


In [27]:
STRADDLE = fns.straddle_sizing_table(_sp)
print("`vega_neutral` -- the note's funded-straddle construction, ANALYTIC ONLY:")
print(STRADDLE.round(6).to_string(index=False))
print("""
NOT RUN, and the reason is measured rather than asserted.
`strat1_curve_gamma.build_backtest` raises on `trade_straddle=True` -- the
swaption leg was never wired into the engine -- so producing a vega_neutral P&L
would mean adding an untested engine path to answer an OPTIONAL part of the
question. It is also not worth it, and the measurement says why: the straddle
whose premium exactly funds the package's own 1-year carry is a median of
$4.17 of underlying DV01 for 20Yx5Y/25Yx5Y and $937 for 10Yx10Y/20Yx10Y against
a $100,000 package -- 0.004% and 0.94% of it. Calling a book with a
0.004%-of-size straddle bolted on "vega-neutral" would be calling an unhedged
flattener vega-neutral.

Only 5Y/30Y funds a real straddle (median $17,169, 17.2% of the package), and
that is precisely because its carry is large and negative (-3.60 bp median) --
which is the same fact, seen from the carry side, that makes it the directional
book this notebook has just taken apart. The one structure where vega_neutral
would be a meaningful construction is the one where the convexity thesis has
already failed on its own terms.""")

`vega_neutral` -- the note's funded-straddle construction, ANALYTIC ONLY:
      structure  straddle_dv01_median  straddle_dv01_mean  straddle_dv01_max  frac_of_package_dv01_median  carry_roll_bp_median  atmf_vol_bp_yr_median  runnable                                       reason
         5Y/30Y          17169.292254        23208.330440       71323.658251                     0.171693             -3.599166               79.63995     False build_backtest raises on trade_straddle=True
        30Y/50Y           2303.211333         2617.505369        6699.266020                     0.023032              1.495226               79.63995     False build_backtest raises on trade_straddle=True
  20Yx5Y/25Yx5Y              4.166758            6.181869          48.734207                     0.000042             -0.000483               79.63995     False build_backtest raises on trade_straddle=True
10Yx10Y/20Yx10Y            936.907015         1146.445217        4276.795523                     0.009

## 13. Statistics — what actually clears

Two haircuts, both measured on **these** books rather than inherited:

* **overlap in time** — monthly cohorts held a year share ~92% of their window,
  so 79 closed cohorts are `span / horizon` non-overlapping observations;
* **overlap across structures** — `k_eff = k / (1 + (k−1)·r̄)` at the measured
  mean pairwise cohort-P&L correlation.

`r̄` is re-measured per sizing, not inherited from the incumbent: a sizing that
changes the leg ratios changes what the four structures have in common, and
assuming it did not would be assuming the answer to a question this study asks.

In [28]:
NEFF = {}
for _sz in fns.SIZINGS:
    NEFF[_sz] = fns.n_eff_report({_l: BK[(_l, _sz)] for _l, _f, _b in fa.STRAT1_STRUCTURES},
                                 cfg=S1)
print(pd.DataFrame({_sz: {k: v for k, v in _n.items()
                          if k in ("n_eff_per_structure_mean", "mean_pairwise_r",
                                   "min_pairwise_r", "max_pairwise_r",
                                   "k_eff_structures", "n_eff_pooled",
                                   "n_common_closed_cohorts", "n_nominal_pooled")}
                    for _sz, _n in NEFF.items()}).T.round(4).to_string())
N_EFF_POOLED = float(NEFF["dv01_neutral"]["n_eff_pooled"])
N_TRIALS = len(CFG.scored_sizings)
BAR = tw.expected_max_sharpe_under_null(N_TRIALS, n_obs=int(round(N_EFF_POOLED)))
print(f"\nincumbent pooled n_eff = {N_EFF_POOLED:.3f} on {NEFF['dv01_neutral']['n_nominal_pooled']} "
      f"nominal cohort-rows; mean pairwise r = {NEFF['dv01_neutral']['mean_pairwise_r']:.3f}")
print(f"E[max Sharpe | null] over {N_TRIALS} distinct sizings at n_eff = "
      f"{N_EFF_POOLED:.2f}: {BAR:.4f} per observation")
print("\nThe bar is set at the INCUMBENT's pooled n_eff, which is the conservative")
print("choice: three of the four sizings measure a LOWER cross-structure")
print("correlation and therefore a higher n_eff, which would lower the bar.")

                   n_eff_per_structure_mean  mean_pairwise_r  min_pairwise_r  max_pairwise_r  n_common_closed_cohorts  k_eff_structures  n_eff_pooled  n_nominal_pooled
dv01_neutral                         7.5017           0.6882          0.4823          0.8572                     79.0            1.3052        9.7915             364.0
pc1_neutral                          7.5017           0.7600          0.5408          0.9329                     79.0            1.2195        9.1487             364.0
pc12_neutral                         7.5017           0.5944          0.4193          0.9014                     79.0            1.4372       10.7814             364.0
slope_beta_hedged                    7.5017           0.5888          0.1197          0.8873                     79.0            1.4460       10.8472             364.0

incumbent pooled n_eff = 9.791 on 364 nominal cohort-rows; mean pairwise r = 0.688
E[max Sharpe | null] over 4 distinct sizings at n_eff = 9.79: 0.3327 per obs

In [29]:
SCORE = fns.sharpe_scoreboard(
    STATS.reset_index()[["structure", "sizing", "sharpe_per_trade", "n_closed",
                         "total_net_bp", "hit_rate", "mtm_sharpe_ann", "skew",
                         "kurtosis"]],
    N_EFF_POOLED, n_trials=N_TRIALS)
print(SCORE.sort_values("sharpe_per_trade", ascending=False).to_string(
    index=False, float_format=lambda v: f"{v:,.4f}"))
CLEARS = SCORE.loc[SCORE["clears_max_null"], ["structure", "sizing"]]
print(f"\nBooks clearing E[max|null] = {BAR:.4f}:")
print(CLEARS.to_string(index=False) if len(CLEARS) else "  NONE")

      structure            sizing  sharpe_per_trade  n_closed  total_net_bp  hit_rate  mtm_sharpe_ann    skew  kurtosis  e_max_sharpe_null  clears_max_null  deflated_sharpe
  20Yx5Y/25Yx5Y slope_beta_hedged            1.0364        79      662.9605    0.8101          0.4992  0.1444    2.4863             0.3327             True           0.9705
  20Yx5Y/25Yx5Y      dv01_neutral            0.9333        79      479.6376    0.7975          0.3471  0.2689    2.5820             0.3327             True           0.9576
  20Yx5Y/25Yx5Y      pc12_neutral            0.9288        79      399.9754    0.8228          0.3116  0.6116    3.9679             0.3327             True           0.9579
  20Yx5Y/25Yx5Y       pc1_neutral            0.5451        79      426.2914    0.7342          0.2771  1.6565    6.9645             0.3327             True           0.8070
        30Y/50Y slope_beta_hedged            0.5078        79      621.8759    0.5949          0.5040  0.2736    1.9310             0.3

**Read the scoreboard honestly.** Seven of sixteen books clear the bar, and the
count flatters them:

* **Four are the same structure.** All four sizings of 20Yx5Y/25Yx5Y clear
  (0.545–1.036). Its *incumbent* already cleared at 0.9333, so what clears is
  the structure, not the re-sizing. Counting them as four independent survivals
  is precisely the error `k_eff = 1.31` exists to prevent.
* **Two of the remaining three are `slope_beta_hedged`** (30Y/50Y 0.508,
  10Yx10Y/20Yx10Y 0.414), and section 7 showed 43% and 47% of those books' P&L
  is the overlay itself — a short-slope position that paid because slope was
  what the flatteners were losing on. Causally sized and honestly costed, but a
  bet on the sample; and on 10Yx10Y/20Yx10Y the hedge ratio is estimated at
  `t = 0.39` with `R² = 0.010`, i.e. it is not estimated at all. That book
  should be read as an accidental short-slope overlay, not as a hedge.
* **The third is `pc12_neutral` on 30Y/50Y** (0.347 against the incumbent's
  0.311) — a genuine, tiny improvement, well inside noise at `n_eff = 9.8`.

`pc1_neutral` on 5Y/30Y has the fourth-highest total P&L in the table and does
**not** clear (0.260), which is the right outcome — and it is **excluded from
every claim** on the section 9 evidence regardless: a rule whose two estimates
differ by 105% of its own total is not a rule with an edge.

The statement this study supports is therefore narrower than the scoreboard and
worth more than a spurious one:

> On 9.8 effective observations, **no sizing rule is shown to beat the
> incumbent on P&L.** What *is* shown — on a measurement, not on a Sharpe — is
> that the hedges remove the factor they target, that the convexity term
> survives that removal on the three tight pairs at `t = 4.1–4.9`, and that on
> 5Y/30Y there is no convexity earner underneath the slope bet at all.

## 14. Equity curves and the house analytics

`compare_curves` across the sizings, on the closed-cohort books and on the daily
MTM; `trade_dashboard` on the best honest book.

In [30]:
SPAN_YEARS = (pd.Timestamp(IDX[-1]) - pd.Timestamp(IDX[0])).days / 365.25
print(f"span_years = {SPAN_YEARS:.2f}")

COHORT_BOOKS = {}
for _sz in fns.SIZINGS:
    _b = pd.concat([BK[(_l, _sz)] for _l, _f, _b2 in fa.STRAT1_STRUCTURES])
    _b = _b[_b["closed"].to_numpy(bool)].copy()
    _b["exit"] = pd.to_datetime(_b["exit"])
    COHORT_BOOKS[_sz] = _b.sort_values("exit")
compare_curves(COHORT_BOOKS,
               title="All four structures pooled — closed 1-year cohorts by sizing, net (bp)",
               time_col="exit", pnl_col="net_bp", unit="bp", side_col="gate_direction")

span_years = 7.61


In [31]:
MTM_BOOKS = {}
for _sz in fns.SIZINGS:
    _e = sum(EQ[(_l, _sz)] for _l, _f, _b2 in fa.STRAT1_STRUCTURES) / DV01
    _d = _e.diff().dropna()
    MTM_BOOKS[_sz] = pd.DataFrame({"date": _d.index, "pnl": _d.to_numpy()})
compare_curves(MTM_BOOKS,
               title="All four structures pooled — daily MTM by sizing, bp of package DV01",
               time_col="date", pnl_col="pnl", unit="bp")

In [32]:
_b530 = {}
for _sz in fns.SIZINGS:
    _b = BK[("5Y/30Y", _sz)]
    _b = _b[_b["closed"].to_numpy(bool)].copy()
    _b["exit"] = pd.to_datetime(_b["exit"])
    _b530[_sz] = _b.sort_values("exit")
compare_curves(_b530,
               title="5Y/30Y — the book whose P&L WAS the slope bet (closed cohorts, net, bp)",
               time_col="exit", pnl_col="net_bp", unit="bp", side_col="gate_direction")

In [33]:
_tight = {}
for _sz in fns.SIZINGS:
    _b = BK[("20Yx5Y/25Yx5Y", _sz)]
    _b = _b[_b["closed"].to_numpy(bool)].copy()
    _b["exit"] = pd.to_datetime(_b["exit"])
    _tight[_sz] = _b.sort_values("exit")
compare_curves(_tight,
               title="20Yx5Y/25Yx5Y — the book where the edge SURVIVES the hedge (net, bp)",
               time_col="exit", pnl_col="net_bp", unit="bp", side_col="gate_direction")

In [34]:
BEST = ("20Yx5Y/25Yx5Y", "pc1_neutral")
print(f"trade_dashboard on the HEADLINE test for the tight pairs: {BEST}")
_bb = BK[BEST]
_bb = _bb[_bb["closed"].to_numpy(bool)].copy()
_bb["exit"] = pd.to_datetime(_bb["exit"])
trade_dashboard(_bb.sort_values("exit"),
                title=f"{BEST[0]} {BEST[1]} — level-neutral, closed 1-year cohorts (net)",
                span_years=SPAN_YEARS, time_col="exit", pnl_col="net_bp",
                unit="bp", side_col="gate_direction")

trade_dashboard on the HEADLINE test for the tight pairs: ('20Yx5Y/25Yx5Y', 'pc1_neutral')


In [35]:
print("CONTROL — the same structure at the incumbent DV01-neutral sizing:")
_bc = BK[("20Yx5Y/25Yx5Y", "dv01_neutral")]
_bc = _bc[_bc["closed"].to_numpy(bool)].copy()
_bc["exit"] = pd.to_datetime(_bc["exit"])
trade_dashboard(_bc.sort_values("exit"),
                title="20Yx5Y/25Yx5Y dv01_neutral — the incumbent (net)",
                span_years=SPAN_YEARS, time_col="exit", pnl_col="net_bp",
                unit="bp", side_col="gate_direction")

CONTROL — the same structure at the incumbent DV01-neutral sizing:


In [36]:
SUMMARY = pd.DataFrame({
    _sz: summary_stats(COHORT_BOOKS[_sz], span_years=SPAN_YEARS, time_col="exit",
                       pnl_col="net_bp", unit="bp", side_col="gate_direction")
         .set_index("metric")["value"]
    for _sz in fns.SIZINGS})
print(SUMMARY.to_string())
print("""
READ THE 'annualised Sharpe' ROW WITH CARE. `summary_stats` annualises by
scaling the per-trade Sharpe by sqrt(trades / year), which is right for
INDEPENDENT trades. These are not: 1-year holds opened monthly across four
correlated structures, so ~48 "trades per year" are worth about ONE
non-overlapping observation, and the sqrt(48) it multiplies by is very nearly
all of the number. The honest evidence test is the scoreboard in section 13.""")

                              dv01_neutral         pc1_neutral       pc12_neutral  slope_beta_hedged
metric                                                                                              
trades                                 316                 316                316                316
net bp / trade                     +5.0303             +7.8154            +1.0733            +3.7659
gross bp / trade                   +6.0303             +8.8023            +2.3144            +9.0699
cost bp / trade                     1.0000              0.9868             1.2411             5.3040
total net bp                      +1589.56            +2469.67            +339.15           +1190.02
hit rate                             63.9%               59.2%              53.5%              56.0%
avg win / avg loss       +19.575 / -20.742   +26.028 / -18.586  +12.403 / -11.952  +18.429 / -14.906
payoff ratio                         0.944               1.400              1.038          

## 15. Persist and verdict

In [37]:
for (_lab, _sz), _eq in EQ.items():
    _s = f"{fns.safe_leg(_lab)}_{_sz}"
    _eq.to_frame("equity_usd").to_parquet(DATA / f"fns_equity_{_s}.parquet")
    BK[(_lab, _sz)].to_parquet(DATA / f"fns_cohorts_{_s}.parquet", index=False)
STATS.reset_index().to_parquet(DATA / "fns_book_stats.parquet", index=False)
RESID.to_parquet(DATA / "fns_residual_exposure.parquet", index=False)
ATTR_TRADE.to_parquet(DATA / "fns_attribution_trade.parquet", index=False)
ATTR_DAILY.to_parquet(DATA / "fns_attribution_daily.parquet", index=False)
LOOKAHEAD.reset_index().to_parquet(DATA / "fns_lookahead.parquet", index=False)
SCORE.to_parquet(DATA / "fns_scoreboard.parquet", index=False)
FRONTIER.to_parquet(DATA / "fns_hedge_frontier.parquet", index=False)
OVL_SUMMARY.to_parquet(DATA / "fns_overlay_summary.parquet", index=False)
print(f"wrote {2 * len(EQ)} per-book parquets + 9 tables to {DATA}")

wrote 32 per-book parquets + 9 tables to C:\Users\chris\clee\ARBS-cvx\notebooks\data\convexity_rv


In [38]:
VERDICT = fns.verdict(CFG, STATS, SCORE, RESID, ATTR_TRADE,
                      CERT_COMPOSE, NEFF["dv01_neutral"], dominant=DOMINANT)
VERDICT["certification_engine_multileg"] = CERT_ENGINE
VERDICT["lookahead"] = LOOKAHEAD.reset_index().to_dict("records")
VERDICT["overlay"] = OVL_SUMMARY.to_dict("records")
VERDICT["carry_tieout"] = CARRY_TIEOUT.to_dict("records")
VERDICT["headline"] = {
    "question": "after neutralising the dominant factor for that structure, "
                "does the convexity edge survive net of the hedge?",
    "answer": "survives on the three tight forward pairs; dies on 5Y/30Y",
    "tight_pairs_net_bp_retained": {
        s: float(STATS.loc[(s, fns.HEADLINE_SIZING[s]), "total_net_bp"]
                 / STATS.loc[(s, "dv01_neutral"), "total_net_bp"])
        for s in ("30Y/50Y", "20Yx5Y/25Yx5Y", "10Yx10Y/20Yx10Y")},
    "5Y30Y_pc12_net_bp": float(STATS.loc[("5Y/30Y", "pc12_neutral"), "total_net_bp"]),
    "5Y30Y_dv01_net_bp": float(STATS.loc[("5Y/30Y", "dv01_neutral"), "total_net_bp"]),
    "pc1_neutral_5Y30Y_disqualified": True,
    "pc1_neutral_5Y30Y_reason": "walk-forward vs full-sample gap is 105% of its "
                                "own total; the P&L is PCA instability, not edge",
}
(DATA / "fns_verdict.json").write_text(json.dumps(VERDICT, indent=1, default=str))
print(json.dumps(VERDICT["headline"], indent=1, default=str))
print(f"\nwrote {DATA / 'fns_verdict.json'}")

{
 "question": "after neutralising the dominant factor for that structure, does the convexity edge survive net of the hedge?",
 "answer": "survives on the three tight forward pairs; dies on 5Y/30Y",
 "tight_pairs_net_bp_retained": {
  "30Y/50Y": 0.8818836292375736,
  "20Yx5Y/25Yx5Y": 0.888778144785432,
  "10Yx10Y/20Yx10Y": 0.6695456602866678
 },
 "5Y30Y_pc12_net_bp": -739.7893093796151,
 "5Y30Y_dv01_net_bp": 268.8029086886919,
 "pc1_neutral_5Y30Y_disqualified": true,
 "pc1_neutral_5Y30Y_reason": "walk-forward vs full-sample gap is 105% of its own total; the P&L is PCA instability, not edge"
}

wrote C:\Users\chris\clee\ARBS-cvx\notebooks\data\convexity_rv\fns_verdict.json


In [39]:
print("=" * 78)
print("FINAL: sizing by structure, walk-forward weights, net of each hedge's cost")
print("=" * 78)
_f = STATS.reset_index()[["structure", "sizing", "gross_bp_total", "cost_bp_total",
                          "total_net_bp", "hit_rate", "sharpe_per_trade",
                          "t_stat_overlap_adj", "breakeven_cost_bp_leg"]].copy()
_f = _f.merge(RESID[["structure", "sizing", "var_level", "var_slope"]],
              on=["structure", "sizing"], how="left")
_f["headline"] = [("<<" if fns.HEADLINE_SIZING.get(s) == z else "")
                  for s, z in zip(_f["structure"], _f["sizing"])]
_f = _f.sort_values(["structure", "sizing"])
print(_f.round(4).to_string(index=False))
print()
print(f"Certification I  (composition vs 4 stored engine runs): "
      f"terminal gap {CERT_COMPOSE['terminal_gap_pct'].abs().max():.1e}%, "
      f"daily corr {CERT_COMPOSE['corr_daily_changes'].min():.8f}")
print(f"Certification II (3-leg engine run of pc12_neutral 5Y/30Y): "
      f"terminal gap {CERT_ENGINE.get('terminal_gap_pct', float('nan')):.1e}%, "
      f"daily corr {CERT_ENGINE.get('corr_daily_changes', float('nan')):.8f}")
print(f"E[max Sharpe | null], {N_TRIALS} sizings at n_eff {N_EFF_POOLED:.2f}: {BAR:.4f}")
print(f"clears: {len(CLEARS)} of {len(SCORE)} books across "
      f"{CLEARS['structure'].nunique()} structures -- but {int((CLEARS['structure'] == '20Yx5Y/25Yx5Y').sum())} "
      f"of them are 20Yx5Y/25Yx5Y, one bet appearing that many times")
print(f"\nnotebook ran in {time.time() - T_START:.0f}s")

FINAL: sizing by structure, walk-forward weights, net of each hedge's cost
      structure            sizing  gross_bp_total  cost_bp_total  total_net_bp  hit_rate  sharpe_per_trade  t_stat_overlap_adj  breakeven_cost_bp_leg  var_level  var_slope headline
10Yx10Y/20Yx10Y      dv01_neutral        579.0958        79.0000      500.0958    0.6203            0.2684              0.7351                 1.8326     0.6134     0.0021         
10Yx10Y/20Yx10Y      pc12_neutral        417.6498        80.7484      336.9014    0.5190            0.1957              0.5360                 1.2931     0.0859     0.0677         
10Yx10Y/20Yx10Y       pc1_neutral        411.1081        76.2711      334.8370    0.5316            0.1765              0.4834                 1.3475     0.0833     0.2420       <<
10Yx10Y/20Yx10Y slope_beta_hedged       1117.1031       185.3629      931.7402    0.6329            0.4141              1.1343                 3.5351        NaN        NaN         
  20Yx5Y/25Yx5Y     